# 🎙️ مفكرة تقييم موديلات التعرف على الصوت (STT Evaluation Notebook)

**متوافقة مع Google Colab ✅**

هذه المفكرة تتيح لك اختبار وتقييم أي موديل للتعرف على الصوت (مثل Whisper) باستخدام بياناتك الخاصة، واختبار مدى تحمله للضوضاء والمؤثرات الصوتية (Robustness & Noise Testing).

**المميزات:**
- ✅ دعم أي موديل Whisper من HuggingFace أو موديل محلي الصنع
- ✅ رفع بيانات مخصصة (CSV + ملفات صوت أو ملف ZIP) مباشرة في Colab
- ✅ حساب WER و CER والتحليل التفصيلي للأخطاء (Sub, Del, Ins)
- ✅ **اختبار المتانة والضوضاء (Robustness)**: ضوضاء بيضاء (White Noise)، جودة هاتف (Telephone)، تسريع/تبطيء (Speed)، صدى (Reverb)، وخفض الصوت (Volume)
- ✅ مشغّل صوت مدمج للاستماع للأخطاء ومقارنة الصوت مع النويز
- ✅ تصدير النتائج كـ CSV و JSON وتنزيلها تلقائياً

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚙️ الخطوة 0: تثبيت المتطلبات وإعداد البيئة
# ═══════════════════════════════════════════════════════════════
# شغّل هذه الخلية مرة واحدة فقط عند فتح النوتبوك

!pip install -q transformers accelerate torch torchaudio librosa soundfile jiwer datasets tqdm pandas matplotlib scipy

import os, sys, json, time, re, tempfile, zipfile
from pathlib import Path
from dataclasses import asdict, dataclass
from typing import Callable, List, Optional, Sequence, Union
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Audio, HTML
import warnings
warnings.filterwarnings("ignore")

import jiwer
import librosa
from scipy.signal import butter, fftconvolve, sosfilt
from tqdm.auto import tqdm

# التحقق من توفر GPU
import torch
if torch.cuda.is_available():
    print(f"🚀 GPU متاح: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ لا يوجد GPU – سيتم استخدام المعالج (CPU). التقييم سيكون أبطأ.")

print("✅ تم تثبيت وتحميل كل المكتبات بنجاح!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 تعريف أدوات التقييم والمؤثرات الصوتية (مدمجة بالكامل)
# ═══════════════════════════════════════════════════════════════

SR = 16000

# ────── Arabic Normalization ──────

_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
_TATWEEL = "\u0640"
_ALEF_FORMS = re.compile("[إأآٱ]")
_NON_WORD = re.compile(r"[^\w\s]")
_SPACES = re.compile(r"\s+")
_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

@dataclass(frozen=True)
class NormConfig:
    remove_diacritics: bool = True
    unify_alef: bool = True
    unify_ya: bool = True
    unify_ta_marbuta: bool = True
    unify_hamza_seat: bool = False
    map_digits: bool = True
    lowercase_latin: bool = True
    def describe(self):
        return asdict(self)

def normalize_arabic(text, cfg=NormConfig()):
    text = str(text or "").strip()
    if cfg.remove_diacritics:
        text = _DIACRITICS.sub("", text)
    text = text.replace(_TATWEEL, "")
    if cfg.unify_alef:
        text = _ALEF_FORMS.sub("ا", text)
    if cfg.unify_ya:
        text = text.replace("ى", "ي")
    if cfg.unify_ta_marbuta:
        text = text.replace("ة", "ه")
    if cfg.unify_hamza_seat:
        text = text.replace("ؤ", "و").replace("ئ", "ي")
    if cfg.map_digits:
        text = text.translate(_DIGITS)
    if cfg.lowercase_latin:
        text = text.lower()
    text = _NON_WORD.sub(" ", text)
    return _SPACES.sub(" ", text).strip()

# ────── Metrics ──────

def utterance_stats(ref, hyp):
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return {"w_sub": w.substitutions, "w_del": w.deletions, "w_ins": w.insertions, "w_hit": w.hits,
            "c_sub": c.substitutions, "c_del": c.deletions, "c_ins": c.insertions, "c_hit": c.hits}

def score_utterances(refs, hyps):
    rows = [{"ref": r, "hyp": h, **utterance_stats(r, h)} for r, h in zip(refs, hyps) if r]
    df = pd.DataFrame(rows)
    df["w_err"] = df.w_sub + df.w_del + df.w_ins
    df["w_ref_len"] = df.w_sub + df.w_del + df.w_hit
    df["c_err"] = df.c_sub + df.c_del + df.c_ins
    df["c_ref_len"] = df.c_sub + df.c_del + df.c_hit
    df["wer"] = df.w_err / df.w_ref_len
    df["cer"] = df.c_err / df.c_ref_len
    return df

def corpus_rate(df, unit="w"):
    return float(df[f"{unit}_err"].sum() / df[f"{unit}_ref_len"].sum())

def bootstrap_ci(df, unit="w", n_boot=1000, seed=42, alpha=0.05):
    err, length = df[f"{unit}_err"].to_numpy(), df[f"{unit}_ref_len"].to_numpy()
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    rates = err[idx].sum(axis=1) / length[idx].sum(axis=1)
    lo, hi = np.percentile(rates, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

def summarize(df, n_boot=1000, seed=42):
    ref_words = df.w_ref_len.sum()
    lo, hi = bootstrap_ci(df, "w", n_boot, seed)
    return {"n_scored": len(df), "wer": corpus_rate(df, "w"), "wer_ci_low": lo, "wer_ci_high": hi,
            "cer": corpus_rate(df, "c"), "sub_rate": df.w_sub.sum() / ref_words,
            "del_rate": df.w_del.sum() / ref_words, "ins_rate": df.w_ins.sum() / ref_words}

def top_confusions(refs, hyps, k=25):
    counts = Counter()
    for ref, hyp in zip(refs, hyps):
        if not ref: continue
        out = jiwer.process_words(ref, hyp)
        r, h = ref.split(), hyp.split()
        for chunk in out.alignments[0]:
            rs, re_, hs, he = chunk.ref_start_idx, chunk.ref_end_idx, chunk.hyp_start_idx, chunk.hyp_end_idx
            if chunk.type == "substitute":
                counts.update(("sub", a, b) for a, b in zip(r[rs:re_], h[hs:he]))
            elif chunk.type == "delete":
                counts.update(("del", a, "") for a in r[rs:re_])
            elif chunk.type == "insert":
                counts.update(("ins", "", b) for b in h[hs:he])
    rows = [{"type": t, "reference": a, "prediction": b, "count": n} for (t, a, b), n in counts.most_common(k)]
    return pd.DataFrame(rows, columns=["type", "reference", "prediction", "count"])

# ────── Audio Features ──────

def _dbfs(value):
    return float(20 * np.log10(max(float(value), 1e-12)))

def _rms(y):
    return float(np.sqrt(np.mean(np.square(y, dtype=np.float64)) + 1e-12))

def audio_features(y, sr=SR):
    y = np.asarray(y, dtype=np.float32)
    if y.size == 0:
        return {"duration_sec": 0.0, "rms_dbfs": -120.0, "peak_dbfs": -120.0,
                "clipping_ratio": 0.0, "silence_ratio": 1.0, "noise_floor_dbfs_proxy": -120.0}
    frame, hop = int(sr * 0.025), int(sr * 0.010)
    rms = _rms(y)
    frames = librosa.util.frame(y, frame_length=frame, hop_length=hop) if len(y) >= frame else y.reshape(-1, 1)
    frame_rms = np.sqrt(np.mean(np.square(frames, dtype=np.float64), axis=0))
    quiet = frame_rms[frame_rms <= np.percentile(frame_rms, 20)]
    return {"duration_sec": len(y) / sr, "rms_dbfs": _dbfs(rms), "peak_dbfs": _dbfs(np.max(np.abs(y))),
            "clipping_ratio": float(np.mean(np.abs(y) >= 0.999)),
            "silence_ratio": float(np.mean(frame_rms < max(1e-4, rms * 0.03))),
            "noise_floor_dbfs_proxy": _dbfs(np.median(quiet))}

# ────── Failure Flags ──────

_ARABIC_LETTER = re.compile(r"[\u0621-\u064A]")
_ANY_LETTER = re.compile(r"[^\W\d_]")

def max_repeat(tokens, max_ngram=4):
    best = 1 if tokens else 0
    for n in range(1, max_ngram + 1):
        for i in range(len(tokens) - n + 1):
            gram, reps, j = tokens[i:i + n], 1, i + n
            while tokens[j:j + n] == gram:
                reps += 1; j += n
            best = max(best, reps)
    return best

def non_arabic_ratio(text):
    letters = _ANY_LETTER.findall(text)
    if not letters: return 0.0
    return 1 - len(_ARABIC_LETTER.findall(text)) / len(letters)

def failure_flags(ref, hyp, length_ratio_max=2.0, repeat_min=3, foreign_max=0.3):
    ref_len, hyp_words = len(ref.split()), hyp.split()
    ratio = len(hyp_words) / ref_len if ref_len else float("inf")
    return {"flag_empty": not hyp.strip(), "flag_too_long": ratio > length_ratio_max,
            "flag_repetition": max_repeat(hyp_words) >= repeat_min and max_repeat(ref.split()) < repeat_min,
            "flag_foreign_script": non_arabic_ratio(hyp) > foreign_max and non_arabic_ratio(ref) <= foreign_max}

# ────── Audio Perturbations (Noise & Distortions) ──────

def _limit_peak(y):
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    return (y / peak if peak > 1.0 else y).astype(np.float32)

def _add_noise_at_snr(y, noise, snr_db):
    scale = _rms(y) / (10 ** (snr_db / 20)) / _rms(noise)
    return _limit_peak(y + noise * scale)

def white_noise(snr_db):
    return lambda y, rng: _add_noise_at_snr(y, rng.normal(size=y.shape).astype(np.float32), snr_db)

def real_noise(bank, snr_db):
    """Mix a random crop of a real noise recording at the target SNR."""
    def apply(y, rng):
        noise = bank[int(rng.integers(len(bank)))]
        if len(noise) < len(y):
            noise = np.tile(noise, int(np.ceil(len(y) / len(noise))))
        start = int(rng.integers(0, len(noise) - len(y) + 1))
        return _add_noise_at_snr(y, noise[start:start + len(y)], snr_db)
    return apply

def telephone():
    """Narrowband telephony: 300-3400 Hz band, 8 kHz, mu-law quantization."""
    sos = butter(4, [300, 3400], btype="bandpass", fs=SR, output="sos")
    mu = 255.0
    def apply(y, rng):
        y = sosfilt(sos, y).astype(np.float32)
        y = librosa.resample(y, orig_sr=SR, target_sr=8000)
        y = np.clip(y, -1.0, 1.0)
        companded = np.sign(y) * np.log1p(mu * np.abs(y)) / np.log1p(mu)
        companded = np.round(companded * 127) / 127
        y = np.sign(companded) * np.expm1(np.abs(companded) * np.log1p(mu)) / mu
        return librosa.resample(y.astype(np.float32), orig_sr=8000, target_sr=SR).astype(np.float32)
    return apply

def speed(rate):
    return lambda y, rng: librosa.effects.time_stretch(y, rate=rate).astype(np.float32)

def gain(db):
    return lambda y, rng: _limit_peak(y * 10 ** (db / 20))

def reverb(rt60=0.5):
    """Synthetic room: exponentially decaying noise impulse response."""
    def apply(y, rng):
        n = int(SR * rt60)
        t = np.arange(n) / SR
        ir = rng.normal(size=n) * np.exp(-6.9 * t / rt60)
        ir /= np.linalg.norm(ir)
        wet = fftconvolve(y, ir.astype(np.float32))[:len(y)]
        return _limit_peak(wet * (_rms(y) / _rms(wet)))
    return apply

def build_conditions(noise_bank=()):
    """Build dictionary of condition_name -> perturbation function."""
    conditions = {
        "clean": None,
        "white_20dB": white_noise(20),
        "white_10dB": white_noise(10),
        "white_5dB": white_noise(5),
        "telephone": telephone(),
        "speed_0.85": speed(0.85),
        "speed_1.15": speed(1.15),
        "low_volume_-25dB": gain(-25),
        "reverb_0.5s": reverb(0.5),
    }
    if len(noise_bank):
        for snr in (20, 10, 5):
            conditions[f"real_noise_{snr}dB"] = real_noise(noise_bank, snr)
    return conditions

# ────── Dataset Loader ──────

_AUDIO_COLUMNS = {"audio", "audio_path", "path", "file", "wav_path", "file_name", "filename", "audio_file", "wav"}
_TEXT_COLUMNS = {"transcription", "reference", "text", "ref", "sentence", "ground_truth", "label", "transcript"}
_ID_COLUMNS = {"id", "uid", "utterance_id", "sample_id", "file_id", "name"}
_GENDER_COLUMNS = {"gender", "sex", "speaker_gender"}
_AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".webm", ".opus"}

def _find_column(columns, candidates, fallback=None):
    lower_map = {c.lower().strip(): c for c in columns}
    for cand in candidates:
        if cand in lower_map: return lower_map[cand]
    return fallback

def _load_audio(path):
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    return y.astype(np.float32)

def _read_csv_file(path):
    sep = "\t" if Path(path).suffix.lower() == ".tsv" else ","
    return pd.read_csv(path, sep=sep).to_dict("records")

def _read_json_file(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(data, list): return data
    raise ValueError(f"Expected a JSON array at top level in {path}")

def _read_jsonl_file(path):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").strip().splitlines():
        line = line.strip()
        if line: rows.append(json.loads(line))
    return rows

_READERS = {".csv": _read_csv_file, ".tsv": _read_csv_file, ".json": _read_json_file, ".jsonl": _read_jsonl_file}

def _load_from_folder(folder):
    folder = Path(folder)
    audio_files = sorted(p for p in folder.iterdir() if p.suffix.lower() in _AUDIO_EXTS)
    if not audio_files: raise FileNotFoundError(f"No audio files found in {folder}")
    items = []
    for af in audio_files:
        txt_path = af.with_suffix(".txt")
        ref = txt_path.read_text(encoding="utf-8").strip() if txt_path.exists() else ""
        items.append({"audio_path": str(af), "reference_raw": ref, "id": af.stem})
    return items

def _load_from_metadata(meta_path, audio_dir, max_samples, seed):
    meta_path = Path(meta_path)
    reader = _READERS[meta_path.suffix.lower()]
    rows = reader(meta_path)
    if not rows: raise ValueError(f"No rows found in {meta_path}")
    columns = list(rows[0].keys())
    audio_col = _find_column(columns, _AUDIO_COLUMNS)
    text_col = _find_column(columns, _TEXT_COLUMNS)
    id_col = _find_column(columns, _ID_COLUMNS)
    gender_col = _find_column(columns, _GENDER_COLUMNS)
    if text_col is None:
        raise ValueError(f"Cannot find a transcription column in {meta_path}. Available: {columns}")
    if audio_col is None:
        raise ValueError(f"Cannot find an audio path column in {meta_path}. Available: {columns}")
    items = []
    for row in rows:
        items.append({"audio_path": str(row[audio_col]), "reference_raw": str(row[text_col] or ""),
                      "id": str(row.get(id_col, "")) if id_col else "",
                      "gender": str(row.get(gender_col, "unknown")) if gender_col else "unknown"})
    return _finalise(items, audio_dir, max_samples, seed)

def _load_from_zip(zip_path, max_samples, seed):
    tmpdir = tempfile.mkdtemp(prefix="stt_eval_")
    with zipfile.ZipFile(zip_path, "r") as zf: zf.extractall(tmpdir)
    extracted = Path(tmpdir)
    meta_files = list(extracted.rglob("*.csv")) + list(extracted.rglob("*.json")) + \
                 list(extracted.rglob("*.jsonl")) + list(extracted.rglob("*.tsv"))
    if meta_files:
        return _load_from_metadata(meta_files[0], extracted, max_samples, seed)
    for folder in [extracted] + list(extracted.iterdir()):
        if folder.is_dir():
            audio_files = [p for p in folder.rglob("*") if p.suffix.lower() in _AUDIO_EXTS]
            if audio_files:
                return _finalise(_load_from_folder(folder), folder, max_samples, seed)
    raise FileNotFoundError(f"No audio or metadata files found inside {zip_path}")

def _finalise(items, audio_dir, max_samples, seed):
    audio_dir = Path(audio_dir)
    rng = np.random.default_rng(seed)
    if max_samples and max_samples < len(items):
        indices = rng.choice(len(items), size=max_samples, replace=False)
        items = [items[i] for i in sorted(indices)]
    result = []
    for i, item in enumerate(items):
        audio_path_str = item.get("audio_path", "")
        if audio_path_str:
            audio_path = Path(audio_path_str)
            if not audio_path.is_absolute(): audio_path = audio_dir / audio_path
            if not audio_path.exists():
                candidate = audio_dir / audio_path.name
                if candidate.exists(): audio_path = candidate
            y = _load_audio(audio_path)
        elif "audio" in item:
            y = np.asarray(item["audio"], dtype=np.float32)
        else:
            raise FileNotFoundError(f"Cannot find audio for item {i}: {item}")
        result.append({"row_index": i, "id": item.get("id", str(i)),
                       "gender": item.get("gender", "unknown"),
                       "reference_raw": item.get("reference_raw", ""), "audio": y})
    return result

def load_custom_data(data_path, audio_dir=None, max_samples=None, seed=42):
    data_path = Path(data_path)
    if data_path.suffix.lower() == ".zip":
        return _load_from_zip(data_path, max_samples, seed)
    if data_path.is_dir():
        meta_files = [p for p in data_path.iterdir() if p.suffix.lower() in _READERS]
        if meta_files:
            return _load_from_metadata(meta_files[0], audio_dir or data_path, max_samples, seed)
        return _finalise(_load_from_folder(data_path), audio_dir or data_path, max_samples, seed)
    if data_path.suffix.lower() in _READERS:
        return _load_from_metadata(data_path, audio_dir or data_path.parent, max_samples, seed)
    raise ValueError(f"Unsupported data_path: {data_path}")

# ────── ASR Pipeline & Robustness Loop ──────

def build_asr(model_id, device):
    from transformers import pipeline
    return pipeline("automatic-speech-recognition", model=model_id, device=device, chunk_length_s=30)

def transcribe(asr, arrays, batch_size, language):
    kwargs = {"task": "transcribe", **(({"language": language} if language else {}))}
    outputs = asr([{"raw": a, "sampling_rate": SR} for a in arrays], batch_size=batch_size, generate_kwargs=kwargs)
    return [o["text"].strip() for o in outputs]

def evaluate_condition(items, transcribe_fn, perturb, rng, norm_cfg, batch_size):
    """Evaluate model under a single perturbation condition."""
    hyps_raw, audio_seconds, elapsed = [], 0.0, 0.0
    for start in range(0, len(items), batch_size):
        batch = items[start:start + batch_size]
        arrays = [it["audio"] if perturb is None else perturb(it["audio"], rng) for it in batch]
        audio_seconds += sum(len(a) for a in arrays) / SR
        t0 = time.perf_counter()
        hyps_raw.extend(transcribe_fn(arrays))
        elapsed += time.perf_counter() - t0
    refs = [normalize_arabic(it["reference_raw"], norm_cfg) for it in items]
    hyps = [normalize_arabic(h, norm_cfg) for h in hyps_raw]
    scored = score_utterances(refs, hyps)
    keep = [i for i, r in enumerate(refs) if r]
    meta = pd.DataFrame([{k: items[i][k] for k in ("row_index", "id", "gender", "reference_raw")} for i in keep])
    feats = pd.DataFrame([audio_features(items[i]["audio"]) for i in keep])
    flag_df = pd.DataFrame([failure_flags(refs[i], hyps[i]) for i in keep])
    df = pd.concat([meta, feats, scored.reset_index(drop=True), flag_df], axis=1)
    df["prediction_raw"] = [hyps_raw[i] for i in keep]
    return df, elapsed / audio_seconds

def run_conditions(items, transcribe_fn, conditions, norm_cfg, batch_size, seed, n_boot):
    """Evaluate every condition; returns (summary_df, {condition: per-utterance df})."""
    summaries, details = [], {}
    for name in tqdm(list(conditions), desc="Evaluating conditions"):
        rng = np.random.default_rng(seed)
        perturb = conditions[name]
        df, rtf = evaluate_condition(items, transcribe_fn, perturb, rng, norm_cfg, batch_size)
        flag_cols = [c for c in df.columns if c.startswith("flag_")]
        summaries.append({"condition": name, **summarize(df, n_boot, seed), "rtf": rtf,
                          **{f"{c}_rate": float(df[c].mean()) for c in flag_cols},
                          "any_flag_rate": float(df[flag_cols].any(axis=1).mean())})
        details[name] = df
    summary = pd.DataFrame(summaries)
    base = summary.loc[summary.condition == "clean", "wer"]
    if len(base):
        summary["wer_delta_vs_clean"] = summary["wer"] - float(base.iloc[0])
    return summary, details

# ────── Evaluator Container ──────

class EvalResult:
    def __init__(self, summary, utterances, model_name, n_items):
        self.summary = summary
        self.utterances = utterances
        self.model_name = model_name
        self.n_items = n_items
    @property
    def wer(self): return self.summary["wer"]
    @property
    def cer(self): return self.summary["cer"]
    @property
    def rtf(self): return self.summary.get("rtf", 0.0)
    @property
    def worst(self):
        cols = [c for c in ["row_index", "reference_raw", "prediction_raw", "wer"] if c in self.utterances.columns]
        return self.utterances.nlargest(30, "wer")[cols]
    @property
    def confusions(self):
        return top_confusions(self.utterances.ref, self.utterances.hyp)
    def flag_summary(self):
        flag_cols = [c for c in self.utterances.columns if c.startswith("flag_")]
        return {c: float(self.utterances[c].mean()) for c in flag_cols}
    def __repr__(self):
        return f"EvalResult(model={self.model_name!r}, n={self.n_items}, WER={self.wer:.2%}, CER={self.cer:.2%})"

def evaluate_model(items, model="openai/whisper-small", language="arabic",
                   batch_size=8, device=None, norm_cfg=NormConfig(), n_boot=1000, seed=42):
    if callable(model):
        transcribe_fn = model
        model_name = getattr(model, "__name__", "custom_model")
    else:
        model_name = str(model)
        if device is None:
            device = 0 if torch.cuda.is_available() else -1
        asr = build_asr(model_name, device)
        transcribe_fn = lambda arrays: transcribe(asr, arrays, batch_size, language)
    hyps_raw, audio_seconds, elapsed = [], 0.0, 0.0
    for start in tqdm(range(0, len(items), batch_size), desc="Transcribing", unit="batch"):
        batch = items[start:start + batch_size]
        arrays = [it["audio"] for it in batch]
        audio_seconds += sum(len(a) for a in arrays) / SR
        t0 = time.perf_counter()
        hyps_raw.extend(transcribe_fn(arrays))
        elapsed += time.perf_counter() - t0
    rtf = elapsed / audio_seconds if audio_seconds > 0 else 0.0
    refs = [normalize_arabic(it["reference_raw"], norm_cfg) for it in items]
    hyps = [normalize_arabic(h, norm_cfg) for h in hyps_raw]
    scored = score_utterances(refs, hyps)
    keep = [i for i, r in enumerate(refs) if r]
    meta = pd.DataFrame([{k: items[i][k] for k in ("row_index", "id", "gender", "reference_raw")} for i in keep])
    feats = pd.DataFrame([audio_features(items[i]["audio"]) for i in keep])
    flag_df = pd.DataFrame([failure_flags(refs[i], hyps[i]) for i in keep])
    df = pd.concat([meta, feats, scored.reset_index(drop=True), flag_df], axis=1)
    df["prediction_raw"] = [hyps_raw[i] for i in keep]
    summary_dict = {**summarize(df, n_boot, seed), "rtf": rtf}
    return EvalResult(summary=summary_dict, utterances=df, model_name=model_name, n_items=len(items))

print("✅ تم تحميل جميع أدوات التقييم والمؤثرات بنجاح!")

---
## 1️⃣ اختيار الموديل

حدد الموديل الذي تريد اختباره. يمكنك استخدام:
- **موديل من HuggingFace**: مثل `openai/whisper-small` أو `openai/whisper-large-v3-turbo`
- **موديل محلي**: مسار مجلد على Google Drive أو الجهاز يحتوي على الأوزان والإعدادات
- **موديل مخصص (Custom)**: دالة بايثون خاصة بك (انظر الخلية التالية)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔧 اختر الموديل هنا:
# ═══════════════════════════════════════════════════════════════

# الخيار 1: موديل من HuggingFace (غيّر الاسم حسب حاجتك)
MODEL = "openai/whisper-small"

# الخيار 2: موديل محلي محفوظ على Google Drive أو مجلد محلي
# MODEL = "/content/drive/MyDrive/my_model"

# اللغة (arabic للعربية)
LANGUAGE = "arabic"

# حجم الباتش (قلّله لو الذاكرة مش كافية)
BATCH_SIZE = 8

print(f"📌 الموديل المحدد: {MODEL}")
print(f"📌 اللغة: {LANGUAGE}")
print(f"📌 حجم الباتش: {BATCH_SIZE}")

### 🧪 (اختياري) استخدام موديل محلي الصنع بالكامل

لو عندك موديل مبني بكود خاص (PyTorch, ONNX, Faster-Whisper، أو أي framework آخر)،
عرّف دالة `my_custom_transcribe` في الخلية التالية وفعّلها.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🧪 (اختياري) عرّف دالة الموديل المخصص هنا:
# ═══════════════════════════════════════════════════════════════
# أزل التعليق (#) عن الكود التالي واكتب كود الموديل الخاص بك:

# import your_custom_library
#
# def my_custom_transcribe(audio_arrays):
#     """
#     دالة التعرف على الصوت الخاصة بك.
#     المدخل: قائمة من مصفوفات numpy (16kHz, mono, float32)
#     المخرج: قائمة من النصوص المتوقعة
#     """
#     results = []
#     for audio in audio_arrays:
#         text = your_model.predict(audio, sampling_rate=16000)
#         results.append(text)
#     return results
#
# # لتفعيل الموديل المخصص:
# MODEL = my_custom_transcribe
# print("✅ تم تفعيل الموديل المخصص")

---
## 2️⃣ تحميل البيانات

| الطريقة | الوصف |
|---|---|
| **📤 رفع ملف ZIP من Colab** | رفع ملف ZIP يحتوي على الصوتيات وملف الـ CSV/JSON مباشرة |
| **📤 رفع ملفات منفصلة** | رفع ملف CSV وملفات الصوت منفصلة |
| **📁 مسار محلي / Google Drive** | تحديد مسار مجلد أو ملف CSV على Drive |
| **🌐 بيانات FLEURS** | استخدام بيانات FLEURS المصرية الجاهزة للاختبار السريع |

**أعمدة CSV المدعومة تلقائياً:**
- الصوت: `audio`, `audio_path`, `path`, `file`, `wav_path`, `file_name`
- النص: `transcription`, `reference`, `text`, `sentence`, `ground_truth`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📂 اختر طريقة تحميل البيانات:
# ═══════════════════════════════════════════════════════════════

# الخيار 1: رفع ملف ZIP من جهازك عبر Colab
DATA_SOURCE = "upload_zip"

# الخيار 2: رفع ملفات صوت + CSV منفصلة (أزل التعليق)
# DATA_SOURCE = "upload_files"

# الخيار 3: مسار على Google Drive أو مسار محلي (أزل التعليق)
# DATA_SOURCE = "local"
# DATA_PATH = "/content/drive/MyDrive/my_data/metadata.csv"
# AUDIO_DIR = "/content/drive/MyDrive/my_data/wavs"

# الخيار 4: بيانات FLEURS الجاهزة للاختبار السريع (أزل التعليق)
# DATA_SOURCE = "fleurs"

# الحد الأقصى لعدد العينات (0 = كل العينات)
MAX_SAMPLES = 100

print(f"📌 طريقة التحميل: {DATA_SOURCE}")
print(f"📌 الحد الأقصى للعينات: {MAX_SAMPLES if MAX_SAMPLES else 'الكل'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📥 تحميل البيانات
# ═══════════════════════════════════════════════════════════════

if DATA_SOURCE == "upload_zip":
    from google.colab import files
    print("📤 ارفع ملف ZIP يحتوي على الصوتيات وملف CSV/JSON:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    items = load_custom_data(zip_name, max_samples=MAX_SAMPLES or None, seed=42)
    print(f"\n✅ تم تحميل {len(items)} عينة من {zip_name}")

elif DATA_SOURCE == "upload_files":
    from google.colab import files
    print("📤 ارفع ملف CSV أو JSON (ملف الـ metadata):")
    uploaded_meta = files.upload()
    meta_name = list(uploaded_meta.keys())[0]
    print("\n📤 الآن ارفع ملفات الصوت:")
    uploaded_audio = files.upload()
    audio_tmp = Path("uploaded_audio")
    audio_tmp.mkdir(exist_ok=True)
    for fname, content in uploaded_audio.items():
        (audio_tmp / fname).write_bytes(content)
    items = load_custom_data(meta_name, audio_dir=str(audio_tmp), max_samples=MAX_SAMPLES or None, seed=42)
    print(f"\n✅ تم تحميل {len(items)} عينة")

elif DATA_SOURCE == "local":
    items = load_custom_data(DATA_PATH, audio_dir=AUDIO_DIR if 'AUDIO_DIR' in dir() else None,
                             max_samples=MAX_SAMPLES or None, seed=42)
    print(f"✅ تم تحميل البيانات: {len(items)} عينة")

elif DATA_SOURCE == "fleurs":
    from datasets import Audio as DsAudio, load_dataset
    ds = load_dataset("google/fleurs", "ar_eg", split="test")
    if MAX_SAMPLES:
        ds = ds.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(ds))))
    ds = ds.cast_column("audio", DsAudio(sampling_rate=SR))
    gender_names = ds.features["gender"].names if hasattr(ds.features["gender"], "names") else None
    items = []
    for i, row in enumerate(ds):
        y = np.asarray(row["audio"]["array"], dtype=np.float32)
        if y.ndim == 2: y = y.mean(axis=0 if y.shape[0] <= 8 else 1)
        gender = gender_names[row["gender"]] if gender_names else row["gender"]
        items.append({"row_index": i, "id": row["id"], "gender": gender,
                      "reference_raw": row["transcription"], "audio": y})
    print(f"✅ تم تحميل بيانات FLEURS المصرية: {len(items)} عينة")

else:
    raise ValueError(f"طريقة تحميل غير معروفة: {DATA_SOURCE}")

# عرض أول 5 عينات للمعاينة
preview = pd.DataFrame([{"id": it["id"], "gender": it.get("gender", "unknown"),
                         "reference": it["reference_raw"][:80] + "..." if len(it["reference_raw"]) > 80 else it["reference_raw"],
                         "duration_sec": round(len(it["audio"]) / SR, 2)}
                        for it in items[:5]])
print("\n📋 أول 5 عينات:")
display(preview)

---
## 3️⃣ (اختياري) ربط Google Drive

لو بياناتك أو الموديل محفوظين على Google Drive، شغّل الخلية التالية لربط حسابك.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📁 (اختياري) ربط Google Drive
# ═══════════════════════════════════════════════════════════════
# أزل التعليق وشغّل الخلية لو محتاج Drive

# from google.colab import drive
# drive.mount('/content/drive')
# print("✅ تم ربط Google Drive بنجاح")
# print("📁 الملفات متاحة في: /content/drive/MyDrive/")

---
## 4️⃣ تشغيل التقييم الأساسي (Clean Evaluation)

الخلية التالية ستشغّل الموديل على الصوت الأصلي النظيف وتحسب كافة المقاييس (WER, CER, RTF, ونسب الأخطاء).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 تشغيل التقييم على الصوت الأصلي
# ═══════════════════════════════════════════════════════════════

result = evaluate_model(
    items=items,
    model=MODEL,
    language=LANGUAGE,
    batch_size=BATCH_SIZE,
    norm_cfg=NormConfig(),
    n_boot=1000,
    seed=42,
)

print(f"\n{'='*60}")
print(f"📊 نتائج التقييم للموديل: {result.model_name}")
print(f"{'='*60}")
print(f"  عدد العينات المقيّمة: {result.summary['n_scored']}")
print(f"  WER (نسبة خطأ الكلمات):  {result.wer:.2%}")
print(f"  CER (نسبة خطأ الحروف):   {result.cer:.2%}")
print(f"  RTF (سرعة المعالجة):      {result.rtf:.3f}")
print(f"  نسبة الاستبدال (Sub):     {result.summary['sub_rate']:.2%}")
print(f"  نسبة الحذف (Del):         {result.summary['del_rate']:.2%}")
print(f"  نسبة الإضافة (Ins):       {result.summary['ins_rate']:.2%}")
print(f"  WER CI 95%:                [{result.summary['wer_ci_low']:.2%}, {result.summary['wer_ci_high']:.2%}]")
print(f"{'='*60}")

---
## 5️⃣ لوحة النتائج والرسوم البيانية

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 لوحة الرسوم البيانية
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. توزيع WER لكل جملة
axes[0].hist(result.utterances["wer"], bins=30, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].axvline(result.wer, color="red", linestyle="--", linewidth=2, label=f"Corpus WER = {result.wer:.2%}")
axes[0].set_xlabel("WER per utterance")
axes[0].set_ylabel("Count")
axes[0].set_title("توزيع WER لكل جملة")
axes[0].legend()

# 2. WER مقابل مدة الصوت
axes[1].scatter(result.utterances["duration_sec"], result.utterances["wer"],
                alpha=0.5, s=20, color="#4CAF50")
axes[1].set_xlabel("Duration (sec)")
axes[1].set_ylabel("WER")
axes[1].set_title("WER مقابل مدة الصوت")

# 3. نسب الأخطاء (S/D/I)
error_types = ["Substitution", "Deletion", "Insertion"]
error_values = [result.summary["sub_rate"], result.summary["del_rate"], result.summary["ins_rate"]]
colors = ["#FF9800", "#F44336", "#9C27B0"]
axes[2].bar(error_types, error_values, color=colors, edgecolor="white")
axes[2].set_ylabel("Rate")
axes[2].set_title("تفصيل نوع الأخطاء (S/D/I)")
for i, v in enumerate(error_values):
    axes[2].text(i, v + 0.005, f"{v:.2%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 6️⃣ تحليل الأخطاء ومشغّل الصوت

هنا يتم عرض أسوأ الجمل أداءً (أعلى WER) مع مشغل صوت مدمج للاستماع ومقارنة النص الأصلي بما توقعه الموديل.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔍 أسوأ 10 جمل (الأعلى WER) مع مشغل الصوت
# ═══════════════════════════════════════════════════════════════

worst_n = 10
worst_indices = result.utterances.nlargest(worst_n, "wer").index.tolist()

for rank, idx in enumerate(worst_indices, 1):
    row = result.utterances.loc[idx]
    item_idx = int(row["row_index"])
    item = items[item_idx] if item_idx < len(items) else None

    print(f"\n{'━'*70}")
    print(f"🔴 الترتيب #{rank} | WER = {row['wer']:.2%} | المدة = {row['duration_sec']:.1f}s")
    print(f"📝 النص المرجعي:  {row['reference_raw']}")
    print(f"🤖 نص الموديل:    {row['prediction_raw']}")

    if item is not None:
        display(Audio(data=item["audio"], rate=SR))

---
## 7️⃣ أكثر الكلمات خطأ (Top Confusions)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔄 أكثر الكلمات التي يخطئ فيها الموديل
# ═══════════════════════════════════════════════════════════════

confusions = result.confusions
print("📊 أكثر 25 خطأ تكراراً:")
print("  - sub = استبدال (الموديل سمع كلمة غلط)")
print("  - del = حذف (الموديل أسقط الكلمة)")
print("  - ins = إضافة (الموديل ألّف كلمة من عنده)")
print()
display(confusions)

---
## 8️⃣ مؤشرات الأعطال (Failure Flags)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚩 ملخص مؤشرات الأعطال
# ═══════════════════════════════════════════════════════════════

flags = result.flag_summary()
flag_labels = {
    "flag_empty": "🔇 ناتج فاضي (الموديل ما طلّع أي نص)",
    "flag_too_long": "📏 ناتج أطول بكتير من المرجع",
    "flag_repetition": "🔁 تكرار (الموديل بيعيد نفس الكلام)",
    "flag_foreign_script": "🌍 حروف غير عربية (غلط في تحديد اللغة)",
}

print("🚩 نسبة كل نوع عطل:")
print("-" * 50)
for key, rate in flags.items():
    label = flag_labels.get(key, key)
    bar = "█" * int(rate * 50)
    print(f"  {label}: {rate:.1%}  {bar}")

---
## 🔊 9️⃣ اختبار المتانة والضوضاء (Robustness & Noise Testing)

هذا القسم يختبر قدرة الموديل على فهم الصوت تحت **مختلف أنواع الضوضاء والتشويش**:

| نوع المؤثر / الظرف | الوصف والتأثير |
|---|---|
| **clean** | الصوت الأصلي النظيف بدون أي تعديل |
| **white_20dB** | ضوضاء بيضاء خفيفة (SNR = 20 dB) |
| **white_10dB** | ضوضاء بيضاء متوسطة (SNR = 10 dB) |
| **white_5dB** | ضوضاء بيضاء شديدة وصعبة (SNR = 5 dB) |
| **telephone** | محاكاة صوت التليفون الأرضي/المحمول (فلتر 300-3400 Hz و 8-bit quantization) |
| **speed_0.85** | صوت بطيء بنسبة 15% (كلام متأنٍ) |
| **speed_1.15** | صوت سريع بنسبة 15% (كلام سريع) |
| **low_volume_-25dB** | صوت منخفض جداً وضعيف (-25 dB) |
| **reverb_0.5s** | صدى غرفة واسعة (Room Reverberation مع RT60=0.5s) |

> 💡 **ملاحظة**: يمكنك اختيار كل الظروف أو تحديد ظروف معينة في الخلية التالية لتوفير الوقت.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔧 اختيار ظروف الضوضاء والتشويش للاختبار
# ═══════════════════════════════════════════════════════════════

# يمكنك إزالة أي سطر لو مش عايز تجربه لتسريع الفحص:
SELECTED_CONDITIONS = {
    "clean": None,                    # الصوت الأصلي النظيف
    "white_20dB": white_noise(20),     # ضوضاء خفيفة
    "white_10dB": white_noise(10),     # ضوضاء متوسطة
    "white_5dB": white_noise(5),       # ضوضاء قوية
    "telephone": telephone(),          # جودة مكالمة هاتفية
    "speed_0.85": speed(0.85),         # كلام بطيء
    "speed_1.15": speed(1.15),         # كلام سريع
    "low_volume_-25dB": gain(-25),     # صوت منخفض جداً
    "reverb_0.5s": reverb(0.5),        # صدى غرفة
}

print(f"📌 عدد الظروف المحددة: {len(SELECTED_CONDITIONS)}")
print(f"📌 قائمة الظروف: {list(SELECTED_CONDITIONS.keys())}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 تشغيل اختبار الضوضاء والمتانة (Robustness Evaluation)
# ═══════════════════════════════════════════════════════════════

if callable(MODEL):
    _transcribe_fn = MODEL
else:
    _device = 0 if torch.cuda.is_available() else -1
    _asr = build_asr(MODEL, _device)
    _transcribe_fn = lambda arrays: transcribe(_asr, arrays, BATCH_SIZE, LANGUAGE)

robustness_summary, robustness_details = run_conditions(
    items=items,
    transcribe_fn=_transcribe_fn,
    conditions=SELECTED_CONDITIONS,
    norm_cfg=NormConfig(),
    batch_size=BATCH_SIZE,
    seed=42,
    n_boot=1000,
)

print("\n" + "="*80)
print("📊 نتائج اختبار المتانة ضد النويز والتشويش (Robustness Summary)")
print("="*80)
display_cols = ["condition", "wer", "cer", "rtf"]
if "wer_delta_vs_clean" in robustness_summary.columns:
    display_cols.append("wer_delta_vs_clean")
summary_display = robustness_summary[display_cols].copy()
summary_display["wer"] = summary_display["wer"].apply(lambda x: f"{x:.2%}")
summary_display["cer"] = summary_display["cer"].apply(lambda x: f"{x:.2%}")
summary_display["rtf"] = summary_display["rtf"].apply(lambda x: f"{x:.3f}")
if "wer_delta_vs_clean" in summary_display.columns:
    summary_display["wer_delta_vs_clean"] = summary_display["wer_delta_vs_clean"].apply(
        lambda x: f"+{x:.2%}" if x > 0 else f"{x:.2%}")
display(summary_display)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 رسم بياني لمقارنة أداء الموديل تحت كل نوع نويز
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

conditions_list = robustness_summary["condition"].tolist()
wer_values = robustness_summary["wer"].tolist()
cer_values = robustness_summary["cer"].tolist()

# تلوين الأعمدة حسب نوع المؤثر
bar_colors = []
for c in conditions_list:
    if c == "clean": bar_colors.append("#4CAF50")      # أخضر (نظيف)
    elif "white" in c: bar_colors.append("#F44336")    # أحمر (ضوضاء)
    elif "telephone" in c: bar_colors.append("#FF9800")# برتقالي (هاتف)
    elif "speed" in c: bar_colors.append("#2196F3")    # أزرق (سرعة)
    elif "volume" in c: bar_colors.append("#9C27B0")   # بنفسجي (صوت منخفض)
    elif "reverb" in c: bar_colors.append("#00BCD4")   # سماوي (صدى)
    else: bar_colors.append("#607D8B")

# 1. WER
axes[0].bar(range(len(conditions_list)), wer_values, color=bar_colors, edgecolor="white")
axes[0].set_xticks(range(len(conditions_list)))
axes[0].set_xticklabels(conditions_list, rotation=45, ha="right", fontsize=9)
axes[0].set_ylabel("WER")
axes[0].set_title("مقارنة نسبة خطأ الكلمات (WER) تحت كل ظرف")
for i, v in enumerate(wer_values):
    axes[0].text(i, v + 0.005, f"{v:.1%}", ha="center", fontsize=8, fontweight="bold")

# 2. CER
axes[1].bar(range(len(conditions_list)), cer_values, color=bar_colors, edgecolor="white")
axes[1].set_xticks(range(len(conditions_list)))
axes[1].set_xticklabels(conditions_list, rotation=45, ha="right", fontsize=9)
axes[1].set_ylabel("CER")
axes[1].set_title("مقارنة نسبة خطأ الحروف (CER) تحت كل ظرف")
for i, v in enumerate(cer_values):
    axes[1].text(i, v + 0.005, f"{v:.1%}", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

# ملخص نصي
if "wer_delta_vs_clean" in robustness_summary.columns:
    worst_cond = robustness_summary.loc[robustness_summary["wer"].idxmax()]
    best_cond = robustness_summary.loc[robustness_summary["wer"].idxmin()]
    clean_wer = robustness_summary.loc[robustness_summary.condition == "clean", "wer"].iloc[0]
    print(f"\n📈 أسوأ ظرف تأثيراً على الموديل: {worst_cond['condition']} (WER = {worst_cond['wer']:.2%})")
    print(f"📉 أفضل ظرف: {best_cond['condition']} (WER = {best_cond['wer']:.2%})")
    print(f"💡 الزيادة في نسبة الخطأ بسبب النويز: +{worst_cond['wer'] - clean_wer:.2%}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎧 استمع للفرق: قارن الصوت الأصلي بالصوت بعد تطبيق النويز والتشويش
# ═══════════════════════════════════════════════════════════════

# اختر رقم العينة للاستماع إليها (0 = أول عينة)
SAMPLE_IDX = 0

sample = items[SAMPLE_IDX]
print(f"📝 النص المرجعي: {sample['reference_raw']}")
print("\n🔊 1. الصوت الأصلي النظيف (clean):")
display(Audio(data=sample["audio"], rate=SR))

# تشغيل عينات الصوت مع كل مؤثر
for cond_name, perturb_fn in SELECTED_CONDITIONS.items():
    if cond_name == "clean" or perturb_fn is None:
        continue
    perturbed = perturb_fn(sample["audio"].copy(), np.random.default_rng(42))
    print(f"\n🔊 {cond_name}:")
    display(Audio(data=perturbed, rate=SR))

---
## 🔟 حفظ وتصدير النتائج والتنزيل

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 💾 حفظ وتنزيل كافة النتائج والتقارير
# ═══════════════════════════════════════════════════════════════

output_dir = Path("outputs") / result.model_name.split("/")[-1]
output_dir.mkdir(parents=True, exist_ok=True)

# حفظ الجدول التفصيلي للعينات
utterances_path = output_dir / "utterances.csv"
result.utterances.drop(columns=["ref", "hyp"], errors="ignore").to_csv(
    utterances_path, index=False, encoding="utf-8-sig")

# حفظ أسوأ 30 جملة
result.worst.to_csv(output_dir / "worst_30.csv", index=False, encoding="utf-8-sig")

# حفظ الكلمات الأكثر خطأ
result.confusions.to_csv(output_dir / "top_confusions.csv", index=False, encoding="utf-8-sig")

# حفظ تقرير المتانة والنويز لو تم تشغيله
if "robustness_summary" in dir():
    robustness_summary.to_csv(output_dir / "robustness_summary.csv", index=False, encoding="utf-8-sig")
    print("   📄 robustness_summary.csv")

# حفظ التقرير الكامل كـ JSON
import platform
report = {
    "model": result.model_name,
    "num_examples": result.n_items,
    "num_scored": result.summary["n_scored"],
    "wer": result.wer,
    "cer": result.cer,
    "rtf": result.rtf,
    "sub_rate": result.summary["sub_rate"],
    "del_rate": result.summary["del_rate"],
    "ins_rate": result.summary["ins_rate"],
    "wer_ci_95": [result.summary["wer_ci_low"], result.summary["wer_ci_high"]],
    "flags": result.flag_summary(),
    "python": platform.python_version(),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
if "robustness_summary" in dir():
    report["robustness"] = robustness_summary[["condition", "wer", "cer", "rtf"]].to_dict("records")

(output_dir / "report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n✅ تم حفظ النتائج في: {output_dir}")
print(f"   📄 {utterances_path.name}")
print(f"   📄 worst_30.csv")
print(f"   📄 top_confusions.csv")
print(f"   📄 report.json")

# تنزيل الملفات تلقائياً في بيئة Colab
try:
    from google.colab import files
    print("\n📥 جاري تنزيل ملفات النتائج لجهازك...")
    files.download(str(output_dir / "report.json"))
    files.download(str(utterances_path))
    files.download(str(output_dir / "worst_30.csv"))
    files.download(str(output_dir / "top_confusions.csv"))
    if "robustness_summary" in dir():
        files.download(str(output_dir / "robustness_summary.csv"))
    print("✅ تم بدء التنزيل بنجاح!")
except ImportError:
    print("💡 في بيئة Colab سيتم تنزيل الملفات تلقائياً إلى مجلد التنزيلات بجهازك.")

---
## 🎓 ملاحظات وتوجيهات مهمة

1. **معيار Normalization**: قواعد التطبيع (حذف التشكيل، توحيد الألف والياء والتاء المربوطة) تُطبّق بالتساوي على النص المرجعي وتوقع الموديل لحساب عادل للـ WER.
2. **اختبار المتانة والنويز**: يعطيك رؤية واقعية لكيفية تصرف الموديل في العالم الحقيقي (في الشارع، عبر الهاتف، في بيئات مزعجة).
3. **RTF (Real-Time Factor)**: قيمة أقل من 1.0 تعني أن الموديل أسرع من الزمن الحقيقي (مثلاً 0.2 تعني 10 ثواني صوت تُعالج في ثانيتين).
4. **مقارنة موديلين**: يمكنك تشغيل النوتبوك على موديلين مختلفين ومقارنة عمود `wer_delta_vs_clean` لمعرفة أيهما أكثر صموداً أمام الضوضاء.
5. **تفعيل الـ GPU في Colab**: تأكد من تفعيل `Runtime` ← `Change runtime type` ← `T4 GPU` لتسريع التقييم حتى 10-20 ضعفاً مقارنة بالـ CPU.